# Download Places365 with Torchvision

This notebook downloads the **Places365-Standard** dataset using `torchvision.datasets.Places365` with `download=True`.

For DreamVision 2.0, we want:
- `split="train-standard"`
- `small=True` for the `256x256` images
- optional `val` download for sanity checks or later evaluation


## Why This Approach

Using `torchvision` keeps the download path inside the PyTorch workflow, so we do not need TensorFlow or manual archive handling for the first pass.

The local `torchvision` implementation supports these splits:
- `train-standard`
- `train-challenge`
- `val`
- `test`

We are intentionally choosing **Standard**, not **Challenge**, because it is much lighter and already large enough for background GAN training.


In [ ]:
from __future__ import annotations

from pathlib import Path

import torchvision
from torchvision.datasets import Places365

print("torchvision:", torchvision.__version__)


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_ROOT = PROJECT_ROOT / "data" / "places365_torchvision"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_SPLIT = "train-standard"
DOWNLOAD_SMALL_IMAGES = True
DOWNLOAD_VAL_SPLIT = True

print("Dataset root:", DATA_ROOT)
print("Train split:", TRAIN_SPLIT)
print("small:", DOWNLOAD_SMALL_IMAGES)
print("download val:", DOWNLOAD_VAL_SPLIT)


## What Torchvision Downloads

With `split="train-standard"` and `small=True`, `torchvision` downloads the official MIT archive:
- `train_256_places365standard.tar`

With `split="val"` and `small=True`, it downloads:
- `val_256.tar`

Those files come from the official Places365 MIT host under `http://data.csail.mit.edu/places/places365/`.


In [ ]:
train_dataset = Places365(
    root=str(DATA_ROOT),
    split=TRAIN_SPLIT,
    small=DOWNLOAD_SMALL_IMAGES,
    download=True,
)

print("Train download complete.")
print("Number of train samples:", len(train_dataset))
print("Number of classes:", len(train_dataset.classes))
print("Images directory:", train_dataset.images_dir)


In [ ]:
val_dataset = None
if DOWNLOAD_VAL_SPLIT:
    val_dataset = Places365(
        root=str(DATA_ROOT),
        split="val",
        small=DOWNLOAD_SMALL_IMAGES,
        download=True,
    )
    print("Validation download complete.")
    print("Number of val samples:", len(val_dataset))
    print("Validation images directory:", val_dataset.images_dir)
else:
    print("Skipping validation download.")


In [ ]:
print("Top-level contents:")
for child in sorted(DATA_ROOT.iterdir()):
    print("-", child.name)


## Important Note for Training

`torchvision.datasets.Places365` can read the official archive layout directly, but it does **not** behave like a simple `ImageFolder` tree.

That means for the background GAN notebook, the cleanest next step is to load Places365 using `Places365(...)` directly instead of assuming `train/<class_name>/*.jpg`.

This will save us from manual folder reorganization.


In [ ]:
sample_image, sample_target = train_dataset[0]
print("First sample target:", sample_target)
print("First sample size:", getattr(sample_image, "size", None))


## Next Step

After this notebook succeeds on HiPerGator, the next move should be updating the background GAN training notebook to use `torchvision.datasets.Places365` directly for its dataloader.
